# Who Gets Heard?
*A Human-Centered Dara Study of Community Reporting in Seattle.*

## Research Scope
This study examines reporting channel patterns in Seattle's 2025 Customer Service Request data. It focuses on *Find It Fix It* App and Phone reports across census tracts and explores their associations with neighborhood median household income.

## 1. Data Collection and Quality Check

### 1.1 ACS Population and Income Data
Retrieve the 2020-2024 ACS dara and check for missing values and duplicate census tract IDs.

In [1]:
import pandas as pd
import requests

# ACS dataset API
acs_url = (
    "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/"
    "arcgis/rest/services/CensusTractTop50_5YALL/"
    "FeatureServer/0/query"
)

# Retrieve 2020–2024 ACS data
params = {
    "where": "ACS_VINTAGE='5Y24'",
    "outFields": (
        "GEOID,ACS_VINTAGE,"
        "TOTAL_POPULATION,"
        "MEDIAN_HH_INC_PAST_12MO_DOLLAR"
    ),
    "returnGeometry": "false",
    "resultRecordCount": 2000,
    "f": "json"
}

response = requests.get(acs_url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

if "error" in data:
    raise RuntimeError(data["error"])

if data.get("exceededTransferLimit"):
    raise RuntimeError("The API did not return all records.")

acs = pd.DataFrame(
    [feature["attributes"] for feature in data["features"]]
)

print("Number of records:", len(acs))

print("\nMissing values:")
print(acs.isna().sum())

print("\nDuplicate GEOIDs:")
print(acs["GEOID"].duplicated().sum())

display(acs.head())

Number of records: 494

Missing values:
GEOID                              0
ACS_VINTAGE                        0
TOTAL_POPULATION                   0
MEDIAN_HH_INC_PAST_12MO_DOLLAR    21
dtype: int64

Duplicate GEOIDs:
0


,GEOID,ACS_VINTAGE,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
0,53033000402,5Y24,5365,128310.0
1,53033000403,5Y24,3285,65469.0
2,53033000700,5Y24,4857,95431.0
3,53033003302,5Y24,3826,153523.0
4,53033003601,5Y24,3507,111908.0


In [2]:
# Check population and income values
display(
    acs[
        ["TOTAL_POPULATION",
         "MEDIAN_HH_INC_PAST_12MO_DOLLAR"]
    ].describe()
)

# Check GEOID format
acs["GEOID"] = acs["GEOID"].astype(str)

print("GEOID lengths:")
print(acs["GEOID"].str.len().value_counts())

# Check records with missing income
print("\nRecords with missing income:")
display(
    acs.loc[
        acs["MEDIAN_HH_INC_PAST_12MO_DOLLAR"].isna(),
        ["GEOID", "TOTAL_POPULATION",
         "MEDIAN_HH_INC_PAST_12MO_DOLLAR"]
    ].head(10)
)

,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
count,494.000000,473.000000
mean,4629.900810,130985.822410
std,1296.307837,45572.715752
min,1295.000000,17500.000000
25%,3680.000000,98478.000000
50%,4468.500000,126786.000000
75%,5545.500000,160049.000000
max,8777.000000,249456.000000


GEOID lengths:
GEOID
11    494
Name: count, dtype: int64

Records with missing income:


,GEOID,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
61,53033011802,5167,NaN
106,53033032318,6767,NaN
110,53033032603,4171,NaN
162,53033006200,4193,NaN
210,53033024500,4966,NaN
239,53033032221,6173,NaN
241,53033032321,6925,NaN
250,53033025008,2533,NaN
271,53033032215,5922,NaN
273,53033032320,4474,NaN


### 1.2 Geographic Coverage Check
Compare ACS census tract IDs with Seattle's 2020 census tract boundaries to identify records relevant to the study area.

In [5]:
%pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
import geopandas as gpd

# Load Seattle 2020 census tract boundaries
tract_url = (
    "https://data-seattlecitygis.opendata.arcgis.com/"
    "api/download/v1/items/9075e8c912a24c4b9458af8866c72ae7/"
    "geojson?layers=3"
)

tracts = gpd.read_file(tract_url)

# Ensure GEOIDs have the same format
tracts["GEOID20"] = tracts["GEOID20"].astype(str)
acs["GEOID"] = acs["GEOID"].astype(str)

# Check matching GEOIDs
matched = tracts["GEOID20"].isin(acs["GEOID"])

print("Seattle census tracts:", len(tracts))
print("Matching ACS records:", matched.sum())
print("Unmatched Seattle tracts:", (~matched).sum())

# Preview unmatched tracts
display(tracts.loc[~matched, ["GEOID20"]].head())

Seattle census tracts: 177
Matching ACS records: 177
Unmatched Seattle tracts: 0


,GEOID20


### 1.3 Seattle ACS Data
Keep only ACS records that match Seattle census tracts and check missing population and income values.

In [7]:
# Keep ACS records that match Seattle census tracts
seattle_acs = acs[
    acs["GEOID"].isin(tracts["GEOID20"])
].copy()

print("Seattle ACS records:", len(seattle_acs))

print("\nMissing values:")
print(seattle_acs.isna().sum())

print("\nDuplicate GEOIDs:")
print(seattle_acs["GEOID"].duplicated().sum())

display(seattle_acs.head())

Seattle ACS records: 177

Missing values:
GEOID                             0
ACS_VINTAGE                       0
TOTAL_POPULATION                  0
MEDIAN_HH_INC_PAST_12MO_DOLLAR    3
dtype: int64

Duplicate GEOIDs:
0


,GEOID,ACS_VINTAGE,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
0,53033000402,5Y24,5365,128310.0
1,53033000403,5Y24,3285,65469.0
2,53033000700,5Y24,4857,95431.0
3,53033003302,5Y24,3826,153523.0
4,53033003601,5Y24,3507,111908.0


### 1.4 Check Invalid Values
Check whether population and median household income contain zero or negative values before saving the ACS data.

In [9]:
# Check invalid population values
print("Population <= 0:",
      (seattle_acs["TOTAL_POPULATION"] <= 0).sum())

# Check invalid income values
print("Income <= 0:",
      (seattle_acs["MEDIAN_HH_INC_PAST_12MO_DOLLAR"] <= 0).sum())

# Show census tracts with missing income
print("\nCensus tracts with missing income:")
display(
    seattle_acs.loc[
        seattle_acs["MEDIAN_HH_INC_PAST_12MO_DOLLAR"].isna(),
        ["GEOID", "TOTAL_POPULATION",
         "MEDIAN_HH_INC_PAST_12MO_DOLLAR"]
    ]
)

Population <= 0: 0
Income <= 0: 0

Census tracts with missing income:


,GEOID,TOTAL_POPULATION,MEDIAN_HH_INC_PAST_12MO_DOLLAR
61,53033011802,5167,NaN
162,53033006200,4193,NaN
352,53033004101,3999,NaN


### 1.5 Save ACS Data
Save the ACS data from Seattle's 177 census tracts. Keep missing income values for later analysis.

In [10]:
from pathlib import Path

# Create data folder
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Select fields needed for analysis
acs_final = seattle_acs[
    [
        "GEOID",
        "ACS_VINTAGE",
        "TOTAL_POPULATION",
        "MEDIAN_HH_INC_PAST_12MO_DOLLAR"
    ]
].copy()

# Save as CSV
acs_final.to_csv(
    data_dir / "acs_2024.csv",
    index=False
)

print("Saved:", data_dir / "acs_2024.csv")
print("Number of records:", len(acs_final))

Saved: data/acs_2024.csv
Number of records: 177
